# Transición LCW · `pettrace-training@v4.100` → `lcw-refined@v1`

**Objetivo.** Publicar una copia portable y verificable de la porción LCW refinada que ya usaron los experimentos.

**Entrada inmutable.** `pettrace-training@v4.100`, fingerprint `d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58`.

**Salida.** Dataset público `alessandro54/lcw-refined@v1`, bajo la licencia `apache-2.0` declarada por el publicador de LCW. La tarjeta conserva atribución y advierte que pueden existir derechos de terceros sobre las fotografías.

Este cuaderno no vuelve a limpiar imágenes, no ejecuta YOLO, no entrena y no publica resultados MLflow. La publicación de datos es un flujo separado de los experimentos.


## Fase 1. Recuperar y verificar la entrada

Ejecuta en Colab. El token `HF_TOKEN` se lee desde Colab Secrets y nunca se imprime. Drive conserva tanto `v4.100` como el staging reanudable de la release.


In [ ]:
%pip install -q --no-cache-dir "pettrace-ml==0.2.16"


In [ ]:
import json
import os
from pathlib import Path

from google.colab import drive, userdata
from pettrace import data

EXPECTED_FINGERPRINT = "d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58"
CACHE = Path("/content/datasets/build/pettrace-training")
DRIVE_ROOT = Path("/content/drive/MyDrive/PetTrace")
STAGING = DRIVE_ROOT / "releases/lcw-refined-v1"
REFERENCE = CACHE / "v4.100.json"

if not DRIVE_ROOT.parent.is_dir():
    drive.mount("/content/drive")
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Falta HF_TOKEN en Colab Secrets o el notebook no tiene acceso.")
os.environ["HF_TOKEN"] = token

prepared, provenance = data.load("v4.100", cache_dir=CACHE, drive_root=DRIVE_ROOT)
if prepared.fingerprint != EXPECTED_FINGERPRINT:
    raise ValueError("v4.100 no coincide con el fingerprint congelado.")
if not REFERENCE.is_file():
    raise FileNotFoundError("No se creó la referencia inmutable v4.100.json.")
print(f"Entrada verificada: {prepared.fingerprint} | fotos: {len(prepared.rows):,}")
print(f"Procedencia: {provenance.get('repo')}@{provenance.get('version')}")


## Fase 2. Construir y auditar `lcw-refined@v1`

La herramienta one-shot selecciona únicamente filas LCW, genera shards TAR deterministas por split y escribe hashes, metadata, licencia y procedencia. Rechaza una entrada distinta de `v4.100` y cualquier archivo inesperado en staging. Todavía no modifica Hugging Face.


In [ ]:
import hashlib
import subprocess
import sys
from urllib.request import urlretrieve

TOOL_URL = "https://raw.githubusercontent.com/alessandro54/pettrace-ml/main/tools/publish_hf_dataset.py"
TOOL_SHA256 = "a99664374bd9f433e2fd0eceffdbf946761590def186bdabeb02e51a8e59bc7b"
TOOL = Path("/content/publish_hf_dataset.py")
urlretrieve(TOOL_URL, TOOL)
if hashlib.sha256(TOOL.read_bytes()).hexdigest() != TOOL_SHA256:
    raise ValueError("La herramienta de publicación no coincide con la revisión auditada.")
subprocess.run([sys.executable, str(TOOL), "--prepared", str(REFERENCE),
                "--staging", str(STAGING)], check=True)


In [ ]:
import csv
from IPython.display import Markdown, display

evidence = json.loads((STAGING / "versions/v1/dataset.json").read_text())
with (STAGING / "versions/v1/metadata.csv").open() as stream:
    preview = list(csv.DictReader(stream))[:5]
display(Markdown(
    f"**Release preparada:** `{evidence['source']}@{evidence['release_version']}`  \\n"
    f"**Origen:** `pettrace-training@{evidence['prepared_version']}`  \\n"
    f"**Fotos LCW:** {evidence['photos']:,} · **Identidades:** {evidence['identities']:,}  \\n"
    f"**Splits:** `{evidence['splits']}` · **Shards:** {len(evidence['shards'])}  \\n"
    f"**Licencia:** `{evidence['license']}`"
))
display(preview)


## Fase 3. Publicar y verificar la release

La siguiente celda realiza la única mutación externa del cuaderno. Publica exclusivamente el inventario auditado, comprueba que todos los archivos existan en el commit remoto y recién entonces crea la etiqueta inmutable `v1`. Repetirla con la misma evidencia es seguro.


In [ ]:
from huggingface_hub import HfApi

subprocess.run([sys.executable, str(TOOL), "--prepared", str(REFERENCE),
                "--staging", str(STAGING), "--upload"], check=True)
api = HfApi(token=os.environ["HF_TOKEN"])
remote = api.dataset_info("alessandro54/lcw-refined", revision="v1")
remote_files = api.list_repo_files("alessandro54/lcw-refined", repo_type="dataset", revision=remote.sha)
required = {"README.md", "LICENSE", "versions/v1/dataset.json", "versions/v1/metadata.csv"}
if not required <= set(remote_files):
    raise RuntimeError("La release remota quedó incompleta.")
print(f"Publicada y verificada: https://huggingface.co/datasets/alessandro54/lcw-refined/tree/v1")
print(f"Commit: {remote.sha} | archivos: {len(remote_files):,}")
